# Lista 13 · APIs REST

Consultas a uma **API REST**: fazer o `GET`, conferir o código de status, ler o
JSON, filtrar com parâmetros e sobreviver às falhas — do equipamento que não existe
ao serviço instável.

A célula de preparo põe no ar a **API simulada da Maré Net** e guarda o endereço
dela em `API`. Toda função recebe esse endereço no parâmetro `api`. Se o Colab
reiniciar, rode a célula de preparo de novo.

---

**Como usar este caderno:** cada exercício tem duas células. Na primeira,
escreva a sua solução no lugar do `# TODO`. A segunda tem os testes —
rode-a e ela diz se a sua função está correta. Não altere a célula de teste.

Se um teste falhar, o Python mostra um `AssertionError` apontando a linha:
é aquele caso específico que a sua função ainda não atende.

**Comece pela célula abaixo.** Ela cria os arquivos de exemplo que os
exercícios desta lista leem. Sem ela, os testes falham com
`FileNotFoundError`.

In [ ]:
# simulador: API REST da Maré Net — não precisa ler (é o "servidor" das aulas).
# Sobe um servidor HTTP nesta própria sessão, em http://127.0.0.1:8765, que
# responde em JSON como a API de um sistema de inventário de rede.
import json
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

PORTA_API = 8765

_EQUIPAMENTOS = [
    {"nome": "OLT-CENTRO-01", "tipo": "OLT", "ip": "10.0.1.10", "localidade": "Centro", "em_servico": True},
    {"nome": "OLT-NORTE-02", "tipo": "OLT", "ip": "10.0.2.10", "localidade": "Zona Norte", "em_servico": True},
    {"nome": "OLT-SUL-03", "tipo": "OLT", "ip": "10.0.3.10", "localidade": "Zona Sul", "em_servico": False},
    {"nome": "SWITCH-CENTRO-01", "tipo": "SWITCH", "ip": "10.0.1.20", "localidade": "Centro", "em_servico": True},
    {"nome": "SWITCH-NORTE-02", "tipo": "SWITCH", "ip": "10.0.2.20", "localidade": "Zona Norte", "em_servico": True},
    {"nome": "RADIO-OESTE-01", "tipo": "RADIO", "ip": "10.0.5.10", "localidade": "Zona Oeste", "em_servico": True},
]
_ALARMES = [
    {"id": 101, "equipamento": "OLT-CENTRO-01", "severidade": "CRITICAL", "mensagem": "perda de sinal na porta GPON0/1/3"},
    {"id": 102, "equipamento": "SWITCH-NORTE-02", "severidade": "ERROR", "mensagem": "Interface Gi0/12, changed state to down"},
    {"id": 103, "equipamento": "RADIO-OESTE-01", "severidade": "WARNING", "mensagem": "enlace degradado"},
    {"id": 104, "equipamento": "OLT-CENTRO-01", "severidade": "WARNING", "mensagem": "temperatura acima do limite"},
    {"id": 105, "equipamento": "RADIO-OESTE-01", "severidade": "CRITICAL", "mensagem": "enlace fora do ar"},
]
_chamadas_instavel = [0]


class _Tratador(BaseHTTPRequestHandler):
    def log_message(self, *args):          # sem log na tela
        pass

    def _responde(self, status, corpo):
        dados = json.dumps(corpo, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()
        self.wfile.write(dados)

    def do_GET(self):
        url = urlparse(self.path)
        partes = [p for p in url.path.split("/") if p]
        filtros = {chave: valores[0] for chave, valores in parse_qs(url.query).items()}
        if partes == ["api", "saude"]:
            return self._responde(200, {"status": "ok"})
        if partes == ["api", "equipamentos"]:
            itens = [e for e in _EQUIPAMENTOS
                     if all(str(e.get(k)) == v for k, v in filtros.items())]
            return self._responde(200, {"count": len(itens), "results": itens})
        if len(partes) == 3 and partes[:2] == ["api", "equipamentos"]:
            for e in _EQUIPAMENTOS:
                if e["nome"] == partes[2]:
                    return self._responde(200, e)
            return self._responde(404, {"erro": f"equipamento {partes[2]} não encontrado"})
        if partes == ["api", "alarmes"]:
            itens = [a for a in _ALARMES
                     if all(str(a.get(k)) == v for k, v in filtros.items())]
            return self._responde(200, {"count": len(itens), "results": itens})
        if partes == ["api", "lento"]:
            time.sleep(3)
            return self._responde(200, {"status": "finalmente"})
        if partes == ["api", "instavel"]:
            _chamadas_instavel[0] += 1
            if _chamadas_instavel[0] % 2 == 1:
                return self._responde(503, {"erro": "serviço temporariamente indisponível"})
            return self._responde(200, {"status": "ok"})
        return self._responde(404, {"erro": f"rota {url.path} não existe"})


def inicia_api(porta=PORTA_API):
    """Sobe a API simulada (se ainda não estiver no ar) e devolve o endereço."""
    endereco = f"http://127.0.0.1:{porta}"
    try:
        urllib.request.urlopen(endereco + "/api/saude", timeout=1)
        return endereco                     # já estava no ar
    except OSError:
        pass
    servidor = ThreadingHTTPServer(("127.0.0.1", porta), _Tratador)
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    return endereco

API = inicia_api()
print("API simulada no ar em", API)

### Exercício 01

Escreva `status_de(api, caminho)`, que faz um `GET` em `api + caminho` (com
`timeout=5`) e devolve o **código de status** da resposta.

```python
status_de(API, "/api/equipamentos/OLT-CENTRO-01")   # -> 200
status_de(API, "/api/nao-existe")                   # -> 404
```

`API` é o endereço da API simulada, que a célula de preparo põe no ar.

In [ ]:
import requests

def status_de(api, caminho):
    """Código de status de um GET em api + caminho."""
    # TODO: requests.get(..., timeout=5) e o .status_code da resposta
    pass

In [ ]:
# Célula de teste — Exercício 01
assert status_de(API, '/api/equipamentos/OLT-CENTRO-01') == 200, status_de(API, '/api/equipamentos/OLT-CENTRO-01')
assert status_de(API, '/api/nao-existe') == 404, status_de(API, '/api/nao-existe')
assert status_de(API, '/api/equipamentos/OLT-XYZ') == 404, status_de(API, '/api/equipamentos/OLT-XYZ')
print("Exercício 01: todos os testes passaram!")

### Exercício 02

Escreva `ip_do_equipamento(api, nome)`, que consulta `/api/equipamentos/<nome>` e
devolve o IP — ou `None` se a resposta não for `200`.

In [ ]:
import requests

def ip_do_equipamento(api, nome):
    """IP do equipamento, ou None se ele não existir."""
    # TODO: confira o status_code antes de usar o .json()
    pass

In [ ]:
# Célula de teste — Exercício 02
assert ip_do_equipamento(API, 'OLT-CENTRO-01') == '10.0.1.10', ip_do_equipamento(API, 'OLT-CENTRO-01')
assert ip_do_equipamento(API, 'RADIO-OESTE-01') == '10.0.5.10', ip_do_equipamento(API, 'RADIO-OESTE-01')
assert ip_do_equipamento(API, 'OLT-XYZ') is None, ip_do_equipamento(API, 'OLT-XYZ')
print("Exercício 02: todos os testes passaram!")

### Exercício 03

Escreva `quantos_equipamentos(api)`, que devolve o total de equipamentos da API (o
campo `count` da resposta de `/api/equipamentos`).

In [ ]:
import requests

def quantos_equipamentos(api):
    """Total de equipamentos cadastrados."""
    # TODO: GET em /api/equipamentos e o campo "count" do .json()
    pass

In [ ]:
# Célula de teste — Exercício 03
assert quantos_equipamentos(API) == 6, quantos_equipamentos(API)
print("Exercício 03: todos os testes passaram!")

### Exercício 04

Escreva `nomes_do_tipo(api, tipo)`, que pede à API **só** os equipamentos daquele
tipo (use `params={"tipo": tipo}`) e devolve a lista dos nomes, na ordem da
resposta.

In [ ]:
import requests

def nomes_do_tipo(api, tipo):
    """Nomes dos equipamentos de um tipo, filtrados no servidor."""
    # TODO: requests.get(..., params={"tipo": tipo}, timeout=5); percorra "results"
    pass

In [ ]:
# Célula de teste — Exercício 04
assert nomes_do_tipo(API, 'OLT') == ['OLT-CENTRO-01', 'OLT-NORTE-02', 'OLT-SUL-03'], nomes_do_tipo(API, 'OLT')
assert nomes_do_tipo(API, 'SWITCH') == ['SWITCH-CENTRO-01', 'SWITCH-NORTE-02'], nomes_do_tipo(API, 'SWITCH')
assert nomes_do_tipo(API, 'ROTEADOR') == [], nomes_do_tipo(API, 'ROTEADOR')
print("Exercício 04: todos os testes passaram!")

### Exercício 05

Escreva `fora_de_servico(api)`, que devolve a lista dos nomes dos equipamentos com
`em_servico` falso. Desta vez, filtre **no seu código**, percorrendo todos os
resultados.

In [ ]:
import requests

def fora_de_servico(api):
    """Nomes dos equipamentos fora de serviço."""
    # TODO: GET em /api/equipamentos e o filtro de sempre
    pass

In [ ]:
# Célula de teste — Exercício 05
assert fora_de_servico(API) == ['OLT-SUL-03'], fora_de_servico(API)
print("Exercício 05: todos os testes passaram!")

### Exercício 06

Escreva `mensagens_de(api, equipamento)`, que consulta `/api/alarmes` com o
parâmetro `equipamento` e devolve a lista das **mensagens** dos alarmes dele.

In [ ]:
import requests

def mensagens_de(api, equipamento):
    """Mensagens dos alarmes de um equipamento."""
    # TODO: params={"equipamento": equipamento}
    pass

In [ ]:
# Célula de teste — Exercício 06
assert mensagens_de(API, 'OLT-CENTRO-01') == ['perda de sinal na porta GPON0/1/3', 'temperatura acima do limite'], mensagens_de(API, 'OLT-CENTRO-01')
assert mensagens_de(API, 'SWITCH-CENTRO-01') == [], mensagens_de(API, 'SWITCH-CENTRO-01')
print("Exercício 06: todos os testes passaram!")

### Exercício 07

Escreva `alarmes_por_severidade(api)`, que devolve um dicionário **severidade →
quantidade** com todos os alarmes da API.

In [ ]:
import requests

def alarmes_por_severidade(api):
    """Dicionário severidade -> quantidade de alarmes."""
    # TODO: GET em /api/alarmes e a contagem do capítulo 5
    pass

In [ ]:
# Célula de teste — Exercício 07
assert alarmes_por_severidade(API) == {'CRITICAL': 2, 'ERROR': 1, 'WARNING': 2}, alarmes_por_severidade(API)
print("Exercício 07: todos os testes passaram!")

### Exercício 08

Escreva `consulta_varios(api, nomes)`, que consulta cada nome e devolve **dois
valores**: um dicionário nome → IP dos que deram certo, e a lista dos nomes que
falharam. Use `raise_for_status()` e `try/except requests.RequestException`
**dentro** do laço.

In [ ]:
import requests

def consulta_varios(api, nomes):
    """(dicionário nome -> IP, lista dos que falharam)."""
    # TODO: para cada nome, um try com o GET, o raise_for_status() e o ["ip"];
    #       no except requests.RequestException, guarde o nome nas falhas
    pass

In [ ]:
# Célula de teste — Exercício 08
assert consulta_varios(API, ['OLT-CENTRO-01', 'OLT-XYZ', 'SWITCH-NORTE-02']) == ({'OLT-CENTRO-01': '10.0.1.10', 'SWITCH-NORTE-02': '10.0.2.20'}, ['OLT-XYZ']), consulta_varios(API, ['OLT-CENTRO-01', 'OLT-XYZ', 'SWITCH-NORTE-02'])
assert consulta_varios(API, []) == ({}, []), consulta_varios(API, [])
print("Exercício 08: todos os testes passaram!")

### Exercício 09

Escreva `com_tentativas(api, caminho, maximo)`, que faz o `GET` e, **enquanto** a
resposta for um erro de servidor (código de 500 para cima), tenta de novo — até
`maximo` tentativas. Devolve **dois valores**: o último código de status e quantas
tentativas foram feitas. Erro 4xx **não** se tenta de novo: o pedido está errado, e
repetir não resolve.

A rota `/api/instavel` alterna entre `503` e `200`.

In [ ]:
import requests

def com_tentativas(api, caminho, maximo):
    """(último status, tentativas feitas), repetindo só em erro 5xx."""
    # TODO: for tentativa in range(1, maximo + 1): faça o GET; se o status for
    #       menor que 500, devolva (status, tentativa); depois do laço, devolva
    #       (último status, maximo)
    pass

In [ ]:
# Célula de teste — Exercício 09
assert com_tentativas(API, '/api/equipamentos/OLT-XYZ', 3) == (404, 1), com_tentativas(API, '/api/equipamentos/OLT-XYZ', 3)
assert com_tentativas(API, '/api/equipamentos', 3) == (200, 1), com_tentativas(API, '/api/equipamentos', 3)
status, tentativas = com_tentativas(API, "/api/instavel", 3)
assert status == 200 and tentativas in (1, 2), (status, tentativas)
print("Exercício 09: todos os testes passaram!")

### Exercício 10

Escreva `diferencas(api, ontem)`, que recebe o conjunto de nomes da coleta de ontem
e devolve **dois valores**, cada um uma lista ordenada: os equipamentos que
**apareceram** na API e os que **sumiram** dela.

In [ ]:
import requests

def diferencas(api, ontem):
    """(apareceram, sumiram) entre a coleta de ontem e a API hoje."""
    # TODO: monte o conjunto de hoje a partir de "results" e use as diferenças
    #       de conjuntos do capítulo 6
    pass

In [ ]:
# Célula de teste — Exercício 10
assert diferencas(API, {'OLT-CENTRO-01', 'OLT-NORTE-02', 'OLT-SUL-03', 'SWITCH-CENTRO-01', 'ONU-SUL-4512'}) == (['RADIO-OESTE-01', 'SWITCH-NORTE-02'], ['ONU-SUL-4512']), diferencas(API, {'OLT-CENTRO-01', 'OLT-NORTE-02', 'OLT-SUL-03', 'SWITCH-CENTRO-01', 'ONU-SUL-4512'})
assert diferencas(API, set()) == (['OLT-CENTRO-01', 'OLT-NORTE-02', 'OLT-SUL-03', 'RADIO-OESTE-01', 'SWITCH-CENTRO-01', 'SWITCH-NORTE-02'], []), diferencas(API, set())
print("Exercício 10: todos os testes passaram!")